# 01 — Data Pipeline & Feature Engineering
**P08: Stock Market Sentiment and Price Movement Predictor**

This notebook covers:
1. Download OHLCV data for all 5 tickers with CSV caching.
2. Validate each DataFrame.
3. Display the head of each stock.
4. Plot all closing prices on a single chart.
5. Engineer features: daily returns, rolling volatility, RSI, and MACD.
6. Scale features with StandardScaler using the training split only.
7. Build the next-day direction target without leakage.
8. Handle NaNs after feature computation.
9. Review class balance across stocks.
10. Create sequences and a chronological train/validation/test split.

---
## 0 · Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')

: 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

from src.config import LOOKBACK, SEED
from src.data_pipeline import load_all_stocks
from src.feature_engineering import (
    add_features,
    create_sequences,
    split_data,
)

np.random.seed(SEED)

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 12,
})
print('Setup complete.')

---
## 1 · Download / Load All Stocks

The pipeline caches each ticker as `data/{TICKER}_ohlcv.csv`.  
If the file already exists, it loads from disk instead of re-downloading.

In [ ]:
stock_data = load_all_stocks()
print(f'\nLoaded tickers: {list(stock_data.keys())}')

---
## 2 · Inspect — Head of Each DataFrame

In [ ]:
for ticker, df in stock_data.items():
    print(f'\n{"═"*60}')
    print(f'  {ticker} | Shape: {df.shape} | {df.index[0].date()} → {df.index[-1].date()}')
    print(f'{"═"*60}')
    display(df.head())

---
## 3 · Summary Statistics Table

In [ ]:
summary = []
for ticker, df in stock_data.items():
    summary.append({
        'Ticker':      ticker,
        'Rows':        len(df),
        'Start':       str(df.index[0].date()),
        'End':         str(df.index[-1].date()),
        'Min Close':   round(df['Close'].min(), 2),
        'Max Close':   round(df['Close'].max(), 2),
        'Mean Volume': f"{int(df['Volume'].mean()):,}",
    })
display(pd.DataFrame(summary))

---
## 4 · Closing Price Chart — All 5 Stocks

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for (ticker, df), color in zip(stock_data.items(), colors):
    # Normalize to 100 at start so all stocks are comparable on same scale
    normalized = df['Close'] / df['Close'].iloc[0] * 100
    ax.plot(df.index, normalized, label=ticker, color=color, linewidth=1.5)

ax.set_title('Normalized Closing Prices — All 5 Stocks (Base = 100)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price (Base = 100)')
ax.legend(loc='upper left', fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../results/closing_prices_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/closing_prices_chart.png')

---
## 5 · Feature Engineering

Computed features per stock:
- `daily_return` — percentage change in closing price
- `return_std_20` — 20-day rolling standard deviation of returns (volatility proxy)
- `RSI_14` — Relative Strength Index with 14-period window
- `MACD_12_26_9` — MACD line (12-day EMA minus 26-day EMA)
- `MACDs_12_26_9` — MACD signal line (9-day EMA of MACD)
- `volume_norm` — log-normalized volume
- `target` — 1 if **next day** close > today close, else 0

> ⚠️ **MACDh (histogram) is intentionally excluded** — it is perfectly collinear with MACD minus Signal, adding no new information.

In [ ]:
featured_data = {}

for ticker, df in stock_data.items():
    feat_df = add_features(df.copy())

    # Define the target as next-day direction to avoid lookahead bias.
    feat_df['target'] = (feat_df['Close'].shift(-1) > feat_df['Close']).astype(int)

    # Log-transform volume so it sits on a more stable scale.
    feat_df['volume_norm'] = np.log1p(feat_df['Volume'])

    # Drop the MACD histogram because it is redundant with MACD and signal.
    if 'MACDh_12_26_9' in feat_df.columns:
        feat_df.drop(columns=['MACDh_12_26_9'], inplace=True)

    # Remove raw OHLCV columns after feature construction.
    feat_df.drop(columns=['Open', 'High', 'Low', 'Volume'], inplace=True, errors='ignore')

    # Drop rows that still contain indicator warm-up values or the shifted target NaN.
    before = len(feat_df)
    feat_df.dropna(inplace=True)
    after = len(feat_df)
    print(f'[{ticker}] Rows before dropna: {before} → after: {after}  (dropped {before - after} warm-up rows)')

    featured_data[ticker] = feat_df

print('\nFeature columns:', [c for c in list(featured_data['AAPL'].columns)])

---
## 6 · Feature Statistics

In [ ]:
print('AAPL Feature Statistics (before scaling):')
feature_cols = ['daily_return', 'return_std_20', 'RSI_14', 'MACD_12_26_9', 'MACDs_12_26_9', 'volume_norm']
display(featured_data['AAPL'][feature_cols].describe().round(4))

---
## 7 · RSI and MACD Plots — AAPL

In [ ]:
aapl = featured_data['AAPL'].copy()
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Closing price
axes[0].plot(aapl.index, aapl['Close'], color='#1f77b4', linewidth=1.5)
axes[0].set_title('AAPL Closing Price', fontweight='bold')
axes[0].set_ylabel('Price (USD)')

# RSI
axes[1].plot(aapl.index, aapl['RSI_14'], color='#ff7f0e', linewidth=1.2)
axes[1].axhline(70, color='red',   linestyle='--', linewidth=0.8, label='Overbought (70)')
axes[1].axhline(30, color='green', linestyle='--', linewidth=0.8, label='Oversold (30)')
axes[1].set_title('RSI (14-period)', fontweight='bold')
axes[1].set_ylabel('RSI')
axes[1].legend(fontsize=9)

# MACD
axes[2].plot(aapl.index, aapl['MACD_12_26_9'],  color='#2ca02c', linewidth=1.2, label='MACD')
axes[2].plot(aapl.index, aapl['MACDs_12_26_9'], color='#d62728', linewidth=1.2, label='Signal')
axes[2].axhline(0, color='black', linestyle='-', linewidth=0.5)
axes[2].set_title('MACD (12, 26, 9)', fontweight='bold')
axes[2].set_ylabel('MACD')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../results/rsi_macd_aapl.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to results/rsi_macd_aapl.png')

---
## 8 · Class Balance Analysis

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
colors_map = {0: '#d62728', 1: '#2ca02c'}

for ax, (ticker, df) in zip(axes, featured_data.items()):
    counts = df['target'].value_counts().sort_index()
    bars = ax.bar(['Down (0)', 'Up (1)'], counts.values,
                  color=[colors_map[0], colors_map[1]], edgecolor='white', linewidth=0.8)
    total = counts.sum()
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{count}\n({count/total*100:.1f}%)',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(ticker, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_ylim(0, max(counts.values) * 1.2)

plt.suptitle('Target Class Balance — Price UP vs DOWN per Stock', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class balance chart saved.')

# Print summary
print('\nClass Balance Summary:')
for ticker, df in featured_data.items():
    up   = df['target'].sum()
    down = len(df) - up
    print(f'  {ticker}: UP={up} ({up/len(df)*100:.1f}%)  DOWN={down} ({down/len(df)*100:.1f}%)')

---
## 9 · Feature Scaling — StandardScaler

**Why scaling matters before LSTM:**
- RSI ranges from 0 to 100.
- MACD ranges roughly from -6 to +8.
- Daily returns stay on a much smaller scale.
- Without scaling, RSI can dominate gradient updates and reduce model stability.

**Important:** The scaler is fitted on the **training split only** and then applied to validation and test data.
This prevents leakage from future statistics.

**Scaler is saved to `results/scaler_{ticker}.pkl`** for use in the Streamlit app.

In [ ]:
import os
os.makedirs('../results', exist_ok=True)

FEATURE_COLS = ['daily_return', 'return_std_20', 'RSI_14', 'MACD_12_26_9', 'MACDs_12_26_9', 'volume_norm']

scaled_data = {}
scalers     = {}

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

for ticker, df in featured_data.items():
    n       = len(df)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    df_scaled = df.copy()

    # Fit scaler on train split ONLY — no leakage
    scaler = StandardScaler()
    df_scaled.iloc[:n_train, df_scaled.columns.get_indexer(FEATURE_COLS)] = \
        scaler.fit_transform(df.iloc[:n_train][FEATURE_COLS])

    # Apply to val and test using train statistics
    df_scaled.iloc[n_train:, df_scaled.columns.get_indexer(FEATURE_COLS)] = \
        scaler.transform(df.iloc[n_train:][FEATURE_COLS])

    scaled_data[ticker] = df_scaled
    scalers[ticker]     = scaler

    # Save scaler for inference use in Streamlit app
    joblib.dump(scaler, f'../results/scaler_{ticker}.pkl')
    print(f'[{ticker}] Scaler fitted on {n_train} train rows and saved.')

print('\nAll scalers saved to results/')

---
## 10 · Verify Scaling — Before vs After

In [ ]:
ticker = 'AAPL'
print(f'{ticker} — BEFORE scaling:')
display(featured_data[ticker][FEATURE_COLS].describe().round(4))

print(f'\n{ticker} — AFTER scaling (train portion should have mean≈0, std≈1):')
n_train = int(len(scaled_data[ticker]) * TRAIN_RATIO)
display(scaled_data[ticker].iloc[:n_train][FEATURE_COLS].describe().round(4))

---
## 11 · Sequence Creation & Chronological Split

- Lookback window = 30 days (from `config.py`)
- Split ratios: 70% train / 15% validation / 15% test
- Split is strictly chronological — **NO shuffling**
- Output shapes: X → (samples, 30, num_features), y → (samples,)

In [ ]:
all_sequences = {}

for ticker, df in scaled_data.items():
    X, y = create_sequences(df[FEATURE_COLS + ['target']], lookback=LOOKBACK)
    splits = split_data(
        X, y, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO
    )
    all_sequences[ticker] = splits
    X_train, X_val, X_test = splits['X_train'], splits['X_val'], splits['X_test']
    y_train, y_val, y_test = splits['y_train'], splits['y_val'], splits['y_test']
    print(f'[{ticker}]  X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')
    print(f'         y_train: {y_train.shape}  y_val: {y_val.shape}  y_test: {y_test.shape}')
    print()

---
## 12 · Save Sequences to Disk

Sequences are saved as `.npy` files in `results/` so that `02_lstm_model.ipynb`  
and `04_fusion_evaluation.ipynb` can load them directly without reprocessing.

In [ ]:
for ticker, seqs in all_sequences.items():
    for split_name, arr in seqs.items():
        path = f'../results/{ticker}_{split_name}.npy'
        np.save(path, arr)
    print(f'[{ticker}] All 6 splits saved to results/')

print('\nData pipeline complete. Ready for LSTM training in 02_lstm_model.ipynb')

---
## Implementation Notes

| Decision | Rationale |
|---|---|
| Target uses `shift(-1)` | Keeps the label aligned with next-day direction without leakage |
| Feature scaling uses `StandardScaler` | Fit on the training split only for consistent scaling |
| `dropna()` is applied after feature construction | Removes indicator warm-up rows and shifted-target NaNs |
| `log1p(Volume)` is used as `volume_norm` | Stabilizes the volume feature scale |
| `MACDh` is excluded | Avoids a redundant collinear feature |
| Class balance chart is included | Makes the target distribution easy to review |
| Scaler is saved for inference | Supports downstream use in the Streamlit app |